# Level the AGAP airborne gravity data

From the BAS data page: "Free air anomaly data was statistically levelled to give a consistent grid. The correction applied is Level_cor and the final anomaly channel is FAA_level."

In [ ]:
# %load_ext autoreload
# %autoreload 2


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import verde as vd

import airbornegeo

## Load data

This is a subset of the BAS AGAP survey over Antarctica's Gamburtsev Subglacial Mountains. The file is download and subset in the notebook `AGAP_gravity_survey`, and the BAS processing steps are repeated in the notebook `processing_AGAP_gravity_survey`.

In [ ]:
data_df = pd.read_csv("../data/AGAP_gravity_survey_processed.csv")
# for testing, limit number of lines
# data_df = data_df[(data_df.line < 40) | (data_df.line.between(85,89))]
# data_df = data_df[(data_df.line <= 5) | (data_df.line == 8) | (data_df.line==85)]

print(data_df.columns)
data_df.head()

In [ ]:
# Wrap data into a Survey object
survey = airbornegeo.Survey(
    data_df,
    line_column="line",
    distance_column="distance_along_line",
)
survey

In [ ]:
airbornegeo.plotly_points(
    survey.data[::10],  # plot every 10th point
    color_col="line",
    robust=False,
    size=3,
)

In [ ]:
# define flight lines vs tie lines with column 'tie' which is True or False
data_df["tie"] = False
data_df.loc[data_df.line >= 84, "tie"] = True

In [ ]:
data_df[data_df.tie].line.sort_values().unique()

In [ ]:
data_df[~data_df.tie].line.sort_values().unique()

## Calculate intersections of lines and ties

In [ ]:
# calculate theoretical intersection points
survey.create_intersection_table(
    method="groups",
    cutoff_dist=2e3,  # if intersection is more than 2km from nearest line or tie data, it is not included
    buffer_dist=500,  # if a theorectical intersection would exist at this distance from the end of a line, include it.
)
survey.intersections

In [ ]:
# since the theorectical intersection points are not exactly on the flight lines, we
# need to interpolate the data values at the intersection points
# as this uses a cubic interpolation on each of the intersection points, it can
# take a few minutes to run.
# It will warn you of the 500 m interpolation window is too small (not enough data points
# to interpolate) for some intersection points, and it will automatically double the
# window size until it has enough data points to interpolate.
survey.interpolate_intersections(
    to_interp=[
        "grav_disturbance_filt",
        "grav_disturbance_filt_level",
        "height",
        "unixtime",
    ],
    window_width=500,
    method="cubic",
    # extrapolate=True,
)

In [ ]:
# airbornegeo.inspect_intersections(
#     data_df,
#     plot_variable=["grav_disturbance_filt", "height"],
# )

In [ ]:
# get list of lines without ties
survey.lines_without_intersections()

In [ ]:
survey.plot_line_and_crosses(
    y=["grav_disturbance_filt", "height"],
    x="distance_along_line",
    line=8,
    y_axes=[1, 2],
    plot_inters=True,
)

## Calculate intersection mistie values

In [ ]:
survey.calculate_crossover_errors(
    data_col="grav_disturbance_filt_level",
    # plot_map=True,
)
inters_bas = survey.intersections

In [ ]:
survey.calculate_crossover_errors(
    data_col="grav_disturbance_filt",
    # plot_map=True,
)

In [ ]:
airbornegeo.plotly_points(
    survey.data[::50],
    color_col="height",
    hover_cols=["line"],
)

In [ ]:
survey.intersections["inter_height_diff"] = np.abs(
    survey.intersections["flight_height"] - survey.intersections["tie_height"]
)
airbornegeo.plotly_points(
    survey.intersections,
    color_col="inter_height_diff",
    hover_cols=["line", "tie"],
    robust=False,
    size=4,
)

In [ ]:
eqs = survey.eq_sources_1d(
    data_column="grav_disturbance_filt",
    depth="default",
    damping=None,
    block_size=500,
)

In [ ]:
survey.update_intersections_with_eq_sources(
    fitted_equivalent_sources=eqs,
    data_column="grav_disturbance_filt",
)
# The updated column is automatically added to survey.data

In [ ]:
survey.plot_line_and_crosses(
    y=["grav_disturbance_filt", "grav_disturbance_filt_eqs", "height"],
    x="distance_along_line",
    line=97,
    y_axes=[1, 1, 2],
    plot_inters=[True, True, True],
)

In [ ]:
survey.calculate_crossover_errors(
    data_col="grav_disturbance_filt_eqs",
    plot_map=True,
)

In [ ]:
up_cont_height = survey.data.height.quantile(0.95)
up_cont_height

In [ ]:
survey.upward_continue_by_line(
    eqs,
    height=up_cont_height,
)
survey.data.head()

In [ ]:
survey.data["upward_continuation_diff"] = (
    survey.data["upward_continued"] - survey.data["grav_disturbance_filt"]
)
survey.plotly_points(
    color_col="upward_continuation_diff",
    hover_cols=["line", "height"],
)

In [ ]:
survey.plotly_points(
    color_col="height",
    hover_cols=["line", "distance_along_line"],
)

In [ ]:
survey.plot_line_and_crosses(
    y=["grav_disturbance_filt", "upward_continued", "height"],
    x="distance_along_line",
    line=95,
    y_axes=[1, 1, 2],
    plot_inters=[True, True, False],
)

In [ ]:
survey.calculate_crossover_errors(
    data_col="upward_continued",
)
# Capture baseline before forking into two independent branches
inters = survey.intersections
data_df = survey.data

## Simple levelling: without weights

In [ ]:
# Create survey instances for each independent branch
# Branch 1: Simple levelling without weights
survey_unweighted = airbornegeo.Survey(
    data_df.copy(),
    line_column="line",
    distance_column="distance_along_line",
)
survey_unweighted.intersections = inters.copy()
survey_unweighted.intersections.head()

In [ ]:
# NEED TO MAKE IT SO THE BELOW FUNCTION CAN BE CALLED MULTIPLE TIMES AND THE LINES ARE RESET BETWEEN CALLS TO THEIR ORIGINAL VALUES

In [ ]:
# # perform a single iteration of levelling
# # choose to level the lines to the ties
# # choose a trend order of 0 (DC shift of lines)

# # lines to ties
# data_df_levelled, inters_levelled = airbornegeo.crossover_pair_levelling(
#     inters_levelled,
#     data_df_levelled,
#     lines_to_level=data_df_levelled[data_df_levelled.tie==False].line.unique(),
#     data_col="grav_disturbance_filt",
#     levelled_col="grav_trend_0",
#     degree=0,
# )
# inters_levelled.head()

In [ ]:
# # TREND 0
# # level lines to ties
# data_df_levelled, inters_levelled = airbornegeo.iterative_line_levelling(
#     inters_levelled,
#     data_df_levelled,
#     iterations=5,
#     lines_to_level=data_df_levelled[data_df_levelled.tie==False].line.unique(),
#     degree=0,
#     data_col="grav_disturbance_filt",
#     levelled_col="grav_trend_0",
#     plot_levelling_convergence=True,
# )

In [ ]:
# TREND 0
# alternative between levelling lines and ties
survey_unweighted.alternating_iterative_line_levelling(
    iterations=5,
    degree=0,
    data_col="grav_disturbance_filt",
    levelled_col="grav_trend_0",
    plot_levelling_convergence=True,
)

In [ ]:
# TREND 1
# alternative between levelling lines and ties
survey_unweighted.alternating_iterative_line_levelling(
    iterations=5,
    degree=1,
    data_col="grav_trend_0",
    levelled_col="grav_trend_1",
    plot_levelling_convergence=True,
)

In [ ]:
plt.hist(survey_unweighted.intersections.mistie)
plt.xlabel("Mistie value")
plt.title(
    f"Histogram of misties; RMSE: {round(airbornegeo.rmse(survey_unweighted.intersections.mistie), 2)}"
)

In [ ]:
# data_df_levelled["levelling_corr"] = data_df_levelled.grav_disturbance_filt - data_df_levelled.grav_trend_1
# airbornegeo.plot_line_and_crosses(
#     data_df_levelled,
#     line=2,
#     x="distance_along_line",
#     y=[
#         "grav_disturbance_filt",
#         "grav_trend_1",
#         "levelling_corr",
#     ],
#     y_axes=[1,1,2],
#     plot_inters=[
#         True,
#         True,
#         False,
#     ],
# )

In [ ]:
survey_unweighted.data["level_diff"] = (
    survey_unweighted.data["grav_trend_1"]
    - survey_unweighted.data["grav_disturbance_filt_level"]
)

airbornegeo.plotly_points(
    survey_unweighted.data[::10],
    color_col="level_diff",
)

In [ ]:
airbornegeo.plotly_points(
    survey_unweighted.data[::10],
    color_col="grav_trend_1",
)

## Grid unlevelled and levelled data

In [ ]:
print(f"{len(survey_unweighted.data)} points before blocking")
data_df_levelled_blocked = airbornegeo.reduce_by_line(
    survey_unweighted.data,
    np.median,
    spacing=500,
    reduce_by="distance_along_line",
)
print(f"{len(data_df_levelled_blocked)} points after blocking")

In [ ]:
import harmonica as hm

coords = (
    data_df_levelled_blocked.easting,
    data_df_levelled_blocked.northing,
    data_df_levelled_blocked.height,
)
damping = 0.1
depth = "default"
block_size = 2e3

eqs_bas_levelled = hm.EquivalentSources(
    damping=damping, depth=depth, block_size=block_size
)
eqs_bas_levelled.fit(coords, data_df_levelled_blocked.grav_disturbance_filt_level)

eqs_unlevelled = hm.EquivalentSources(
    damping=damping, depth=depth, block_size=block_size
)
eqs_unlevelled.fit(coords, data_df_levelled_blocked.grav_disturbance_filt)

eqs_levelled = hm.EquivalentSources(damping=damping, depth=depth, block_size=block_size)
eqs_levelled.fit(coords, data_df_levelled_blocked.grav_trend_1)

In [ ]:
# Define grid coordinates
grid_coords = vd.grid_coordinates(
    region=vd.pad_region(vd.get_region(coords), 20e3),
    spacing=1e3,
    extra_coords=4200,  # upward continue
    adjust="region",
)

grid_bas_levelled = eqs_bas_levelled.grid(grid_coords)
grid_unlevelled = eqs_unlevelled.grid(grid_coords)
grid_levelled = eqs_levelled.grid(grid_coords)

maxdist = 10e3

grid_bas_levelled = vd.distance_mask(
    (coords[0], coords[1]), maxdist=maxdist, grid=grid_bas_levelled
)
grid_unlevelled = vd.distance_mask(
    (coords[0], coords[1]), maxdist=maxdist, grid=grid_unlevelled
)
grid_levelled = vd.distance_mask(
    (coords[0], coords[1]), maxdist=maxdist, grid=grid_levelled
)

grid_bas_levelled = grid_bas_levelled.reset_coords(names="upward").scalars
grid_unlevelled = grid_unlevelled.reset_coords(names="upward").scalars
grid_levelled = grid_levelled.reset_coords(names="upward").scalars

In [ ]:
# _ = ptk.grid_compare(
#     grid_unlevelled,
#     grid_levelled,
#     grid1_name="Unlevelled",
#     grid2_name="Levelled",
#     hist=True,
# )
# _ = ptk.grid_compare(
#     grid_bas_levelled,
#     grid_levelled,
#     grid1_name="BAS levelled",
#     grid2_name="Levelled",
#     hist=True,
# )

In [ ]:
# titles = [
#     "Unlevelled derivatives",
#     "Levelled derivatives",
#     "BAS levelled derivatives",
# ]
# for i, g in enumerate(
#     [
#         grid_unlevelled,
#         grid_levelled,
#         grid_bas_levelled,
#     ]
# ):
#     east_deriv = airbornegeo.filter_grid(
#         g,
#         filter_type="easting_deriv",
#     )
#     north_deriv = airbornegeo.filter_grid(g, filter_type="northing_deriv")
#     up_deriv = airbornegeo.filter_grid(
#         g,
#         filter_type="up_deriv",
#     )
#     total_gradient = airbornegeo.filter_grid(
#         g,
#         filter_type="total_gradient",
#     )

#     fig = ptk.subplots(
#         [east_deriv, north_deriv, up_deriv, total_gradient],
#         fig_title=titles[i],
#         titles=[
#             "Easting derivative",
#             "Northing derivative",
#             "Upward derivative",
#             "Total gradient",
#         ],
#         cmap="balance+h0",
#         robust=True,
#         fig_height=10,
#     )
#     fig.show()

## Level with automatical intersection weights

In [ ]:
# Branch 2: Levelling with automatic intersection weights
# Reset to baseline before this independent branch
survey_weighted = airbornegeo.Survey(
    data_df.copy(),
    line_column="line",
    distance_column="distance_along_line",
)
survey_weighted.intersections = inters.copy()
survey_weighted.intersections.head()

In [ ]:
survey_weighted.filter_line(
    filter_width=20000,
    data_column="grav_disturbance_filt",
    filter_by_column="distance_along_line",
    result_column="grav_filt",
)
survey_weighted.data["grav_1st_derive"] = np.abs(
    survey_weighted.data.groupby("line")
    .apply(
        lambda x: pd.Series(
            np.gradient(x.grav_filt, x.distance_along_line), index=x.index
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
)
survey_weighted.filter_line(
    filter_width=20000,
    data_column="grav_1st_derive",
    filter_by_column="distance_along_line",
    result_column="grav_1st_derive",
)

survey_weighted.data["grav_2nd_derive"] = np.abs(
    survey_weighted.data.groupby("line")
    .apply(
        lambda x: pd.Series(
            np.gradient(x.grav_1st_derive, x.distance_along_line), index=x.index
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
)
survey_weighted.filter_line(
    filter_width=20000,
    data_column="grav_2nd_derive",
    filter_by_column="distance_along_line",
    result_column="grav_2nd_derive",
)
survey_weighted.data.head()

In [ ]:
survey_weighted.plot_line_and_crosses(
    y=[
        "grav_disturbance_filt",
        "grav_1st_derive",
        "grav_2nd_derive",
    ],
    x="distance_along_line",
    line=5,
    y_axes=[
        1,
        2,
        3,
    ],
    plot_inters=[False, True, True],
)

In [ ]:
survey_weighted.data["height_1st_derive"] = np.abs(
    survey_weighted.data.groupby("line")
    .apply(
        lambda x: pd.Series(
            np.gradient(x.height, x.distance_along_line), index=x.index
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
)
survey_weighted.filter_line(
    filter_width=20000,
    data_column="height_1st_derive",
    filter_by_column="distance_along_line",
    result_column="height_1st_derive",
)

survey_weighted.data["height_2nd_derive"] = np.abs(
    survey_weighted.data.groupby("line")
    .apply(
        lambda x: pd.Series(
            np.gradient(x.height_1st_derive, x.distance_along_line), index=x.index
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
)
survey_weighted.filter_line(
    filter_width=20000,
    data_column="height_2nd_derive",
    filter_by_column="distance_along_line",
    result_column="height_2nd_derive",
)
survey_weighted.data.head()

In [ ]:
survey_weighted.plot_line_and_crosses(
    y=[
        "height",
        "height_1st_derive",
        "height_2nd_derive",
    ],
    x="distance_along_line",
    line=5,
    y_axes=[
        1,
        2,
        3,
    ],
    plot_inters=[False, True, True],
    # use_intersection_y=False,
)

### Create weights for each intersection
When fitting polynomials to each line's mistie values, we can weight certain intersection more or less. This may depend on several factors, such as: 
* distance of intersection to actual data
* if the intersection was interpolated or extrapolated
* if the intersection is at a location of high gradient magnetic data or flight elevations
* if there is a large elevation difference between the flight and the tie at the intersection

We convert these factors into numeric values, where higher values mean the intersection is reliable. 

Should these weights be calculated based on percentile rank for the entire dataset, or just per line? 

In [ ]:
# inters_levelled_weighted = airbornegeo.calculate_intersection_weights(
#     survey_weighted.intersections,
#     survey_weighted.data,
#     weight_by="all",
#     max_dist_weight=1,
#     height_difference_weight=1,
#     height_col_name="height",
#     interpolation_type_weight=1,
#     data_1st_derive_weight=1,
#     data_1st_derive_col_name="grav_1st_derive",
#     data_2nd_derive_weight=1,
#     data_2nd_derive_col_name="grav_2nd_derive",
#     height_1st_derive_weight=1,
#     height_1st_derive_col_name="height_1st_derive",
#     height_2nd_derive_weight=1,
#     height_2nd_derive_col_name="height_2nd_derive",
#     plot=True,
# )
# survey_weighted.intersections = inters_levelled_weighted
# survey_weighted.intersections.sort_values(by="mistie_weight", ascending=False)


In [ ]:
# # TREND 0
# # alternative between levelling lines and ties
# data_df_levelled_weighted, inters_levelled_weighted = (
#     airbornegeo.iterative_line_levelling_alternate(
#         inters_levelled_weighted,
#         data_df_levelled_weighted,
#         iterations=5,
#         degree=0,
#         data_col="grav_disturbance_filt",
#         levelled_col="grav_trend_0",
#         intersection_weight_col="mistie_weight",
#         plot_levelling_convergence=True,
#     )
# )

In [ ]:
# # TREND 1
# # alternative between levelling lines and ties
# data_df_levelled_weighted, inters_levelled_weighted = (
#     airbornegeo.iterative_line_levelling_alternate(
#         inters_levelled_weighted,
#         data_df_levelled_weighted,
#         iterations=5,
#         degree=1,
#         data_col="grav_trend_0",
#         levelled_col="grav_trend_1",
#         intersection_weight_col="mistie_weight",
#         plot_levelling_convergence=True,
#     )
# )

In [ ]:
# plt.hist(inters_levelled_weighted.mistie)
# plt.xlabel("Mistie value")
# plt.title(
#     f"Histogram of misties; RMSE: {round(airbornegeo.rmse(inters_levelled_weighted.mistie), 2)}"
# )

In [ ]:
# data_df_levelled_weighted["level_diff"] = (
#     data_df_levelled_weighted["grav_trend_1"]
#     - data_df_levelled_weighted["grav_disturbance_filt_level"]
# )

# airbornegeo.plotly_points(
#     data_df_levelled_weighted[::10],
#     color_col="level_diff",
# )

In [ ]:
# airbornegeo.plotly_points(
#     data_df_levelled_weighted[::10],
#     color_col="grav_trend_1",
# )

In [ ]:
# airbornegeo.plotly_points(
#     data_df_levelled_weighted[::10],
#     color_col="grav_disturbance_filt",
# )